# 📦 Structured Output & Parsers — Hands-On Examples

> **Module 03 | Submodule 03 | Practical Notebook**
>
> Sections:
> 1. Why structured output matters
> 2. StrOutputParser — token streaming
> 3. JsonOutputParser — flexible dicts
> 4. JSON mode — model-enforced JSON
> 5. PydanticOutputParser — typed validation
> 6. `.with_structured_output()` — the modern way
> 7. Real-world extraction pipelines
> 8. Enum classification

In [ ]:
!pip install langchain langchain-openai pydantic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "module-03-structured-output"

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Ready!")

---
## 1️⃣ The Problem — Unstructured Output

In [ ]:
# Ask the LLM the same question 3 times WITHOUT structure
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("human", "Extract name and age from: {text}")
])

samples = (
    (prompt | ChatOpenAI(model="gpt-4o-mini", temperature=1))
    .batch([{"text": "Alice is 30 years old."}] * 4)
)

print("Same question, 4 runs WITHOUT structure:")
for i, r in enumerate(samples, 1):
    print(f"Run {i}: {r.content!r}")

# Notice: 4 completely different formats!

In [ ]:
# Trying to write code that handles all those formats = nightmare
for r in samples:
    try:
        # This will fail on most formats
        name = r.content.split("name is ")[1].split("."  )[0]
        print(f"Parsed name: {name}")
    except (IndexError, ValueError) as e:
        print(f"Failed to parse: {r.content!r} → {e}")

---
## 2️⃣ StrOutputParser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human",  "{question}")
])

# Without StrOutputParser
chain_raw = prompt | llm
result_raw = chain_raw.invoke({"question": "What is LangChain?"})
print("Without parser:")
print(f"  type: {type(result_raw).__name__}")   # AIMessage
print(f"  value: {result_raw}")
print()

# With StrOutputParser
chain_str = prompt | llm | StrOutputParser()
result_str = chain_str.invoke({"question": "What is LangChain?"})
print("With StrOutputParser:")
print(f"  type: {type(result_str).__name__}")  # str
print(f"  value: {result_str[:80]}...")

In [ ]:
# Streaming — token by token
from langchain_openai import ChatOpenAI

streaming_chain = (
    ChatPromptTemplate.from_template("Write a 3-sentence explanation of {topic}.")
    | ChatOpenAI(model="gpt-4o-mini")
    | StrOutputParser()
)

print("Streaming (token by token):")
for chunk in streaming_chain.stream({"topic": "LangChain"}):
    print(chunk, end="", flush=True)
print()  # newline

---
## 3️⃣ JsonOutputParser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", """You extract information into JSON.
Always return valid JSON only — no explanation, no markdown.
Schema: {{"name": string, "age": integer, "job": string}}"""),
    ("human", "Extract from: {text}")
])

parser = JsonOutputParser()
chain = prompt | llm | parser

texts = [
    "John Smith is a 35-year-old data scientist.",
    "Maria, 28, works as an ML engineer at OpenAI.",
    "Bob Chen (42) is the VP of Engineering at Google.",
]

for text in texts:
    result = chain.invoke({"text": text})
    print(f"Input: {text}")
    print(f"  type: {type(result).__name__}")
    print(f"  name: {result.get('name')}")
    print(f"  age: {result.get('age')} (type: {type(result.get('age')).__name__})")
    print()

In [ ]:
# Streaming JSON — partial output builds up
print("Streaming JSON (partial objects):")
for partial in chain.stream({"text": "Alice Wang, 31, senior Python developer"}):
    print(partial)

---
## 4️⃣ JSON Mode (Model-Enforced)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser

# JSON mode — model is FORCED to return valid JSON
llm_json_mode = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}}  # ← JSON mode
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract technical skills as JSON. Return: {\"skills\": [\"...\", ...]}"),
    ("human",  "{bio}")
])

chain = prompt | llm_json_mode | JsonOutputParser()

bio = """
Carlos is a backend developer with 6 years experience.
He's proficient in Python, FastAPI, PostgreSQL, Docker, and Kubernetes.
He also knows LangChain and has built several RAG systems.
"""

result = chain.invoke({"bio": bio})
print("Skills extracted:", result["skills"])

---
## 5️⃣ PydanticOutputParser

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

# Define typed schema
class TechProfile(BaseModel):
    name: str = Field(description="Person's full name")
    years_experience: int = Field(description="Total years of experience")
    primary_language: str = Field(description="Primary programming language")
    frameworks: List[str] = Field(description="Frameworks and tools they know")
    seniority: str = Field(description="junior, mid, senior, or lead")
    looking_for_job: Optional[bool] = Field(default=None, description="True/false if mentioned")

parser = PydanticOutputParser(pydantic_object=TechProfile)

# See the format instructions generated
print("Format instructions preview (first 300 chars):")
print(parser.get_format_instructions()[:300])

In [ ]:
# Build chain with format instructions injected
prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract developer profile information.\n{format_instructions}"),
    ("human",  "{text}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

profiles = [
    "Alice has 7 years coding Python. She knows FastAPI, LangChain, and PyTorch. She's a senior developer.",
    "Junior dev Bob, 1 year experience with JavaScript and React. Actively looking for a new job.",
    "Lead engineer Sarah, 12 years exp. Specializes in Go and Kubernetes. Not looking.",
]

for profile in profiles:
    result = chain.invoke({"text": profile})
    print(f"Profile: {profile[:50]}...")
    print(f"  → name: {result.name}")
    print(f"  → type: {type(result).__name__}       ← Pydantic model!")
    print(f"  → years_experience: {result.years_experience} ({type(result.years_experience).__name__})")
    print(f"  → seniority: {result.seniority}")
    print(f"  → frameworks: {result.frameworks}")
    print()

---
## 6️⃣ `.with_structured_output()` — Modern Approach

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    director: Optional[str] = Field(default=None, description="Director name if mentioned")
    rating: int = Field(description="Rating out of 10")
    genres: List[str] = Field(description="Movie genres")
    sentiment: str = Field(description="positive, negative, or mixed")
    one_liner: str = Field(description="One-sentence summary of the review")

# The modern way — no parser class, no format instructions!
structured_llm = llm.with_structured_output(MovieReview)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract movie review information."),
    ("human",  "{review}")
])

chain = prompt | structured_llm  # No parser at the end!

reviews = [
    "Inception (directed by Nolan) is an absolute masterpiece! 10/10. Mind-bending sci-fi thriller.",
    "The movie was okay, maybe 6/10. Had some good moments but the ending was disappointing. A drama/thriller.",
]

for review in reviews:
    result = chain.invoke({"review": review})
    print(f"Review: {review[:60]}...")
    print(f"  type:       {type(result).__name__}")
    print(f"  title:      {result.title}")
    print(f"  rating:     {result.rating}/10")
    print(f"  sentiment:  {result.sentiment}")
    print(f"  one_liner:  {result.one_liner}")
    print()

In [ ]:
# include_raw=True — get raw AIMessage AND parsed result
structured_llm_raw = llm.with_structured_output(MovieReview, include_raw=True)
chain_raw = prompt | structured_llm_raw

result = chain_raw.invoke({"review": "Parasite is a brilliant social thriller, 9/10!"})

print("Keys in result:", list(result.keys()))  # raw, parsed, parsing_error
print("Raw type:",    type(result["raw"]).__name__)     # AIMessage
print("Parsed type:", type(result["parsed"]).__name__)  # MovieReview
print("Parsed:", result["parsed"])

---
## 7️⃣ Enum-Based Classification

In [ ]:
from pydantic import BaseModel, Field
from enum import Enum
from typing import List

class Category(str, Enum):
    billing = "billing"
    technical = "technical"
    shipping = "shipping"
    account = "account"
    other = "other"

class Priority(str, Enum):
    urgent = "urgent"
    high = "high"
    medium = "medium"
    low = "low"

class SupportTicket(BaseModel):
    summary: str = Field(description="One-line summary")
    category: Category = Field(description="Ticket category")
    priority: Priority = Field(description="Priority level based on urgency and impact")
    requires_human: bool = Field(description="True if needs human agent, false if bot can handle")
    key_issue: str = Field(description="Core problem in 5 words or less")

structured_llm = llm.with_structured_output(SupportTicket)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify this customer support ticket."),
    ("human",  "{ticket}")
])
chain = prompt | structured_llm

tickets = [
    "I was charged twice for my subscription this month! This is unacceptable.",
    "Hi, how do I change my password?",
    "My package has been stuck in transit for 2 weeks and it contains medication I need.",
    "The AI chatbot on your site is giving wrong answers about pricing.",
]

print(f"{'Ticket':<50} {'Category':<12} {'Priority':<10} {'Human?'}")
print("-" * 85)
for ticket in tickets:
    result = chain.invoke({"ticket": ticket})
    human_icon = "🧑" if result.requires_human else "🤖"
    print(f"{ticket[:48]:<50} {result.category.value:<12} {result.priority.value:<10} {human_icon}")

---
## 8️⃣ Real-World Pipeline — Resume Parser

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class WorkExperience(BaseModel):
    company: str
    title: str
    years: Optional[float] = None

class Resume(BaseModel):
    full_name: str = Field(description="Candidate full name")
    email: Optional[str] = Field(default=None, description="Email address if present")
    total_experience_years: float = Field(description="Total years of professional experience")
    skills: List[str] = Field(description="All technical skills mentioned")
    work_history: List[WorkExperience] = Field(description="Work experience, most recent first")
    education: str = Field(description="Highest degree and field")
    suitable_for_senior_role: bool = Field(
        description="True if 5+ years experience and strong technical skills"
    )

structured_llm = llm.with_structured_output(Resume)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Parse this resume and extract all information accurately."),
    ("human",  "{resume_text}")
])
chain = prompt | structured_llm

resume_text = """
Jane Doe | jane.doe@email.com

EXPERIENCE:
- Senior ML Engineer, Netflix (2021-2024, 3 years)
  Built recommender systems using PyTorch and Spark
- ML Engineer, Spotify (2018-2021, 3 years)
  Developed audio classification models
- Data Scientist, Startup XYZ (2016-2018, 2 years)

SKILLS: Python, PyTorch, TensorFlow, Spark, SQL, LangChain, Docker, AWS

EDUCATION: MS Computer Science, Stanford University
"""

result = chain.invoke({"resume_text": resume_text})

print(f"Name:          {result.full_name}")
print(f"Email:         {result.email}")
print(f"Experience:    {result.total_experience_years} years")
print(f"Skills:        {', '.join(result.skills)}")
print(f"Education:     {result.education}")
print(f"Senior role:   {'✅ Yes' if result.suitable_for_senior_role else '❌ No'}")
print("\nWork History:")
for job in result.work_history:
    print(f"  • {job.title} at {job.company} ({job.years}y)")

---
## ✅ Parser Comparison Summary

| Parser | Returns | Validates | Streams | Use When |
|---|---|---|---|---|
| `StrOutputParser` | `str` | ❌ | ✅ Best | Text output, display to users |
| `JsonOutputParser` | `dict` | ❌ | ✅ | Flexible/unknown JSON schema |
| `PydanticOutputParser` | Pydantic obj | ✅ | ❌ | Fixed schema, need types |
| `.with_structured_output()` | Pydantic obj | ✅ | ✅ | **Always prefer this** |

**Next**: [04 — Chains & Runnables](../04_chains_and_runnables/theory.md)